# Stage A

The detector ask every time , what's the last Id and Because the ID's insert of the first group are committed later than the second group maybe because latency , the second group Id's are read by the detector as the latest so when the first group is commited the detector is not going to read them because it only reads the latest ID's .for example group of id's 1,2,3,4,5 are inserted and then group of id's 6,7,8,9,10 are inserted and the second group is commited before the first group so when the detector ask for the latest id's it will read 10 and when the first group is commited it will not read them because it only reads the latest ID but very fast , is going to receive that the last id is 10 and after the 1,2,3,4,5 are commited they are just going to be ignored by the detector.

## 2. Full plant (60 machines, 600 readings/s)

The plant keeps up (~601 readings/s into PostgreSQL), but the detector does not.
The model costs 3 ms per reading, so one detector processes at most ~333 readings/s
— about half of what arrives. While the plant ran:

- `backlog` grew continuously, to 28,479 rows after ~90 s (the detector read only
  29,628 of 53,880 readings);
- `delay` grew by ~3 s every report, to 47.5 s — far above the 5 s requirement (R3);
- the fault on M-029 was detected (anomalies from seq 479 on, rising score), but
  only ~47 s after the readings were taken: too late for the maintenance team.

`missing` stayed at 32: because the detector is ~45 s behind, it only reads rows
that were committed long ago, so the commit-order race from step 1 hardly occurs.
The slowness hides the completeness problem; a detector that kept up would lose
readings again as in step 1.

# Stage B

60 machines × 10 Hz = 600 readings/s in the lab; rollout to 2,000 machines × 50 Hz
= 100,000 readings/s. The model is stateful per machine and costs 3 ms per reading.
Stage A showed two problems with today's system: [readings skipped because of
commit order (4,900 of 25,540 missing)] and [the detector falls behind (delay 47 s
after 90 s at 600 readings/s)].

| Requirement | Option 1: PostgreSQL, better polling | Option 2: Kafka |
|---|---|---|
| R1 Scale | ⚠️ more pollers help, but every poller queries the DB; at 100,000/s the DB is the bottleneck | ✅ scales with partitions + detector instances |
| R2 Complete & ordered | ❌ `id > last_id` still skips rows committed out of order | ✅ key = `machine_id` → one partition per machine, in order; offsets assigned at write, nothing skipped |
| R3 Fresh (≤ 5 s) | ⚠️ polling adds delay; more pollers = more DB load | ✅ consumers read continuously; enough instances keep lag ≈ 0 |
| R4 Independent consumers | ❌ archive/dashboard = more pollers on the same DB | ✅ each gets its own consumer group, no load on others or on PostgreSQL |

In summary kafka wins.

Therefore I decide to use kafka and use 3 instances of the detector to process the readings and send the anomalies to the anomalies topic.
the gateways write every reading to a Kafka topic readings, key = machine_id. Detector instances share the topic in one consumer group; the archive and dashboard use their own groups.

One detector instance: 1 / 3 ms ≈ 333 readings/s.
- Lab, 600 readings/s: 600 / 333 ≈ 1.8 → 2 instances, 3 with extra room;
  topic with 6 partitions.
- Rollout, 100,000 readings/s: 100,000 / 333 ≈ 300 → ~300 instances, ~400 with
  extra room; at least as many partitions

# Scorer Report

Run 1794399796: started 13:23:16 (5.4 min ago), 60 machines x 10 Hz, 5 faults so far

Detection  (topic anomalies: 1,767 anomaly events, models ['v1'])
   #  machine   start  detected   delay
   1  M-029       40s       yes   7.9 s
   2  M-026      100s       yes   8.2 s
   3  M-004      160s       yes   8.4 s
   4  M-036      220s       yes   7.4 s
   5  M-043      280s       yes   8.1 s

  faults detected: 5/5, mean delay 8.0 s; false alarms (episodes on healthy machines): 0
  latency reading -> anomaly event: p50 0.09 s, p95 0.12 s, max 0.14 s
  duplicate anomaly events (same anomaly_id): 0

Detector instances  (model telemetry, this run)
  det-1          pid 17272   v1  readings    75,034  missing       0  out-of-order      0  machines   25  silent for 23 s
  det-2          pid 30724   v1  readings    60,037  missing       0  out-of-order      0  machines   20  silent for 23 s
  det-3          pid 31440   v1  readings    45,040  missing       0  out-of-order      0  machines   15  silent for 23 s

Requirements (measurable part)
  [PASS] R2 complete and ordered: 0 readings missing, 0 out of order
  [PASS] R3 fresh: p95 latency < 5 s
  [PASS] detection: all faults found, at most 2 false alarms


*Therefore the score was satisfactory*